# DBConformer v2 — Acoustic Model Training

**Three training phases:**
- **Phase 1** — train from scratch with OneCycleLR (100 epochs)
- **Phase 2** — continue from best Phase 1 checkpoint with a lower OneCycleLR peak (50 epochs)
- **Phase 3** — stable fine-tuning with ReduceLROnPlateau (100 epochs)

**Workflow:** Edit the ` Configuration` cell → Run All → checkpoints save automatically to Drive.

Each phase picks up from the best checkpoint of the previous one. You can run only Phase 1 if you want a quick baseline.

# Dependency

In [ ]:
!pip install -q einops timm h5py

In [ ]:
import argparse
import gc
import glob
import os
import pickle
import random
import shutil
import warnings

import h5py
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import Tensor
from torch.backends import cudnn
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Sampler, Dataset as TorchDataset
from einops import rearrange
from timm.models.layers import trunc_normal_
from tqdm.auto import tqdm
from google.colab import drive

warnings.filterwarnings('ignore')
cudnn.benchmark     = False
cudnn.deterministic = True

LOGIT_TO_PHONEME = [
    'BLANK',
    'AA', 'AE', 'AH', 'AO', 'AW', 'AY', 'B',  'CH', 'D',  'DH',
    'EH', 'ER', 'EY', 'F',  'G',  'HH', 'IH', 'IY', 'JH', 'K',
    'L',  'M',  'N',  'NG', 'OW', 'OY', 'P',  'R',  'S',  'SH',
    'T',  'TH', 'UH', 'UW', 'V',  'W',  'Y',  'Z',  'ZH', '|',
]
NUM_PHONEMES = len(LOGIT_TO_PHONEME)  # 41

print('Imports OK | PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())

In [ ]:
drive.mount('/content/drive')

# Data preparation

In [ ]:
# ================================================================
# ⚙️  CONFIGURATION — edit these before running
# ================================================================

# ── Data paths ────────────────────────────────────────────────────
DRIVE_DATA_DIR  = '/content/drive/MyDrive/hdf5_data_final/'
LOCAL_DATA_DIR  = '/content/local_data/hdf5_data_final'
MAX_FILES       = None   # None = load all session files

# ── Checkpoint save directories ───────────────────────────────────
SAVE_DIR_P1     = '/content/drive/MyDrive/DBConformer_runs_512'
SAVE_DIR_P2     = '/content/drive/MyDrive/DBConformer_runs_512_p2'
SAVE_DIR_P3     = '/content/drive/MyDrive/DBConformer_runs_512_p3'

# ── Model hyperparameters ─────────────────────────────────────────
args = argparse.Namespace(
    data_name             = 'NeuralPhoneme',
    session               = 't15.2023.08.18',
    chn                   = 512,
    time_sample_num       = 1500,
    emb_size              = 512,
    spa_dim               = 16,
    transformer_depth_tem = 6,
    transformer_depth_chn = 6,
    temporal_kernel       = 15,
    pool_kernel           = 15,
    pool_stride           = 8,
    gate_flag             = False,
    posemb_flag           = True,
    chn_atten_flag        = True,
    branch                = 'all',
    ffn_hidden            = 1024,
    class_num             = NUM_PHONEMES,
    max_time_steps        = 1500,
    # Training
    dropoutRate           = 0.3,
    lr                    = 1e-3,
    batch_size            = 28,
    max_epoch             = 100,
    eval_interval         = 5,
)

torch.manual_seed(42)
np.random.seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
# ================================================================
# Data pipeline
# ================================================================

def load_h5py_file(file_path: str) -> dict:
    data = {
        'neural_features': [], 'n_time_steps': [], 'seq_class_ids': [],
        'seq_len': [], 'transcriptions': [], 'sentence_label': [],
        'session': [], 'block_num': [], 'trial_num': [],
    }
    with h5py.File(file_path, 'r') as f:
        for key in f.keys():
            g = f[key]
            data['neural_features'].append(g['input_features'][:].astype(np.float32))
            data['n_time_steps'].append(int(g.attrs['n_time_steps']))
            data['seq_class_ids'].append(
                g['seq_class_ids'][:].astype(np.int64).flatten() if 'seq_class_ids' in g else None)
            data['seq_len'].append(int(g.attrs['seq_len']) if 'seq_len' in g.attrs else None)
            data['transcriptions'].append(g['transcription'][:] if 'transcription' in g else None)
            data['sentence_label'].append(
                g.attrs['sentence_label'][:] if 'sentence_label' in g.attrs else None)
            data['session'].append(g.attrs['session'])
            data['block_num'].append(g.attrs['block_num'])
            data['trial_num'].append(g.attrs['trial_num'])
    print(f'  Loaded {len(data["neural_features"])} trials from {file_path}')
    return data


def _merge_dicts(dicts):
    merged = {k: [] for k in dicts[0]}
    for d in dicts:
        for k in merged:
            merged[k].extend(d[k])
    return merged


def load_all_files(drive_dir, local_dir, split, max_files=None):
    local_pattern = os.path.join(local_dir, '**', f'data_{split}.hdf5')
    local_paths   = sorted(glob.glob(local_pattern, recursive=True))
    skip_drive = False
    if local_paths:
        if max_files is None or len(local_paths) >= max_files:
            skip_drive = True
            if max_files is not None:
                local_paths = local_paths[:max_files]
    if skip_drive:
        print(f"Found {len(local_paths)} local files for '{split}'. Skipping Drive.")
    else:
        drive_pattern = os.path.join(drive_dir, '**', f'data_{split}.hdf5')
        drive_paths   = sorted(glob.glob(drive_pattern, recursive=True))
        if not drive_paths:
            raise FileNotFoundError(f"No 'data_{split}.hdf5' found in {drive_dir}")
        if max_files is not None:
            drive_paths = drive_paths[:max_files]
        print(f"Copying {len(drive_paths)} '{split}' files to local disk...")
        local_paths = []
        for dp in drive_paths:
            rel = os.path.relpath(dp, drive_dir)
            lp  = os.path.join(local_dir, rel)
            os.makedirs(os.path.dirname(lp), exist_ok=True)
            if not os.path.exists(lp):
                shutil.copy2(dp, lp)
            local_paths.append(lp)
    print(f"Loading {len(local_paths)} session files...")
    return _merge_dicts([load_h5py_file(p) for p in local_paths])


class SpeechDataset(TorchDataset):
    def __init__(self, data_dict, max_time_steps=1500):
        self.data = data_dict
        self.length = len(data_dict['neural_features'])
        self.max_time_steps = max_time_steps

    def __len__(self):
        return self.length

    def __getitem__(self, idx):
        n_time_steps      = int(self.data['n_time_steps'][idx])
        seq_len           = int(self.data['seq_len'][idx])
        effective_n_steps = min(n_time_steps, self.max_time_steps)
        neural   = self.data['neural_features'][idx][:effective_n_steps]
        phonemes = self.data['seq_class_ids'][idx][:seq_len]
        neural_tensor  = torch.from_numpy(neural).float().transpose(0, 1).unsqueeze(0)
        phoneme_tensor = torch.from_numpy(phonemes.astype(np.int64))
        return neural_tensor, phoneme_tensor, seq_len, effective_n_steps


class BucketBatchSampler(Sampler):
    def __init__(self, dataset, batch_size, drop_last=False):
        self.dataset    = dataset
        self.batch_size = batch_size
        self.drop_last  = drop_last
        self.sorted_indices = sorted(
            range(len(dataset)), key=lambda i: dataset.data['n_time_steps'][i])

    def __iter__(self):
        batches = [self.sorted_indices[i:i + self.batch_size]
                   for i in range(0, len(self.sorted_indices), self.batch_size)]
        random.shuffle(batches)
        for batch in batches:
            if not self.drop_last or len(batch) == self.batch_size:
                yield batch

    def __len__(self):
        if self.drop_last:
            return len(self.dataset) // self.batch_size
        return (len(self.dataset) + self.batch_size - 1) // self.batch_size


def collate_fn(batch):
    neurals, phonemes_list, seq_lens, time_steps = zip(*batch)
    neurals_reshaped = [n.squeeze(0).transpose(0, 1) for n in neurals]
    neural_batch = (pad_sequence(neurals_reshaped, batch_first=True)
                    .transpose(1, 2).unsqueeze(1))
    phoneme_batch   = pad_sequence(phonemes_list, batch_first=True, padding_value=0)
    return (neural_batch,
            phoneme_batch,
            torch.tensor(seq_lens,   dtype=torch.long),
            torch.tensor(time_steps, dtype=torch.long))


def compute_token_lengths(input_time_steps, pool_kernel, pool_stride):
    return ((input_time_steps - pool_kernel) // pool_stride + 1).long()


print('Data pipeline defined.')

# DBConformer

In [ ]:
# ================================================================
# Model: DBConformer
# ================================================================

class Conv(nn.Module):
    def __init__(self, conv, activation=None, bn=None):
        super().__init__()
        self.conv = conv
        self.activation = activation
        if bn:
            self.conv.bias = None
        self.bn = bn

    def forward(self, x):
        x = self.conv(x)
        if self.bn:         x = self.bn(x)
        if self.activation: x = self.activation(x)
        return x


class InterFre(nn.Module):
    def forward(self, x):
        return F.gelu(sum(x))


class Stem(nn.Module):
    def __init__(self, data_name, in_planes, out_planes=64,
                 kernel_size=15, pool_kernel=15, pool_stride=8, radix=2):
        super().__init__()
        self.out_planes = out_planes
        self.mid_planes = out_planes * radix
        self.radix      = radix
        self.sconv = Conv(
            nn.Conv1d(in_planes, self.mid_planes, 1, bias=False, groups=radix),
            bn=nn.BatchNorm1d(self.mid_planes))
        self.tconv = nn.ModuleList()
        ks = kernel_size
        for _ in range(radix):
            self.tconv.append(Conv(
                nn.Conv1d(out_planes, out_planes, ks, 1,
                          groups=out_planes, padding=ks // 2, bias=False),
                bn=nn.BatchNorm1d(out_planes)))
            ks = max(ks // 2, 3)
        self.interFre     = InterFre()
        self.downSampling = nn.AvgPool1d(pool_kernel, pool_stride)
        self.dp           = nn.Dropout(0.5)

    def forward(self, x):
        out = self.sconv(x)
        out = torch.split(out, self.out_planes, dim=1)
        out = [m(xi) for xi, m in zip(out, self.tconv)]
        out = self.interFre(out)
        out = self.downSampling(out)
        return self.dp(out)


class PatchEmbeddingTemporal(nn.Module):
    def __init__(self, data_name, in_planes, out_planes, kernel_size,
                 radix, pool_kernel, pool_stride, time_points, num_classes):
        super().__init__()
        self.stem = Stem(
            data_name=data_name, in_planes=in_planes * radix,
            out_planes=out_planes, kernel_size=kernel_size,
            pool_kernel=pool_kernel, pool_stride=pool_stride, radix=radix)
        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            trunc_normal_(m.weight, std=.01)
            if m.bias is not None: nn.init.constant_(m.bias, 0)
        elif isinstance(m, (nn.LayerNorm, nn.BatchNorm1d, nn.BatchNorm2d)):
            if m.weight is not None: nn.init.constant_(m.weight, 1.0)
            if m.bias   is not None: nn.init.constant_(m.bias, 0)
        elif isinstance(m, (nn.Conv1d, nn.Conv2d)):
            trunc_normal_(m.weight, std=.01)
            if m.bias is not None: nn.init.constant_(m.bias, 0)

    def forward(self, x):
        return self.stem(x).permute(0, 2, 1)  # (B, P, D)


class PatchEmbeddingSpatial(nn.Module):
    def __init__(self, spa_dim, emb_size=40):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv1d(1, spa_dim, kernel_size=25, stride=5, padding=12),
            nn.ELU(),
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
            nn.Linear(spa_dim, emb_size),
        )

    def forward(self, x):
        B, C, T = x.shape
        x = x.reshape(B * C, 1, T)
        x = self.encoder(x)
        return x.view(B, C, -1)


class MultiHeadAttention(nn.Module):
    def __init__(self, emb_size, num_heads, dropout):
        super().__init__()
        assert emb_size % num_heads == 0
        self.num_heads = num_heads
        self.dropout_p = dropout
        self.queries    = nn.Linear(emb_size, emb_size)
        self.keys       = nn.Linear(emb_size, emb_size)
        self.values     = nn.Linear(emb_size, emb_size)
        self.projection = nn.Linear(emb_size, emb_size)

    def forward(self, x, mask=None):
        Q = rearrange(self.queries(x), 'b n (h d) -> b h n d', h=self.num_heads)
        K = rearrange(self.keys(x),    'b n (h d) -> b h n d', h=self.num_heads)
        V = rearrange(self.values(x),  'b n (h d) -> b h n d', h=self.num_heads)
        out = F.scaled_dot_product_attention(
            Q, K, V, attn_mask=mask,
            dropout_p=self.dropout_p if self.training else 0.0, is_causal=False)
        return self.projection(rearrange(out, 'b h n d -> b n (h d)'))


class FeedForwardBlock(nn.Sequential):
    def __init__(self, emb_size, expansion=4, drop_p=0.3):
        super().__init__(
            nn.Linear(emb_size, expansion * emb_size),
            nn.GELU(), nn.Dropout(drop_p),
            nn.Linear(expansion * emb_size, emb_size))


class TransformerEncoderBlock(nn.Module):
    def __init__(self, emb_size, num_heads=8, drop_p=0.3,
                 forward_expansion=4, forward_drop_p=0.3):
        super().__init__()
        self.norm1 = nn.LayerNorm(emb_size)
        self.attn  = MultiHeadAttention(emb_size, num_heads, drop_p)
        self.drop1 = nn.Dropout(drop_p)
        self.norm2 = nn.LayerNorm(emb_size)
        self.ff    = FeedForwardBlock(emb_size, forward_expansion, forward_drop_p)
        self.drop2 = nn.Dropout(drop_p)

    def forward(self, x, mask=None):
        res = x; x = self.norm1(x); x = self.attn(x, mask=mask); x = res + self.drop1(x)
        res = x; x = self.norm2(x); x = self.ff(x);               x = res + self.drop2(x)
        return x


class TransformerEncoder(nn.Module):
    def __init__(self, depth, emb_size):
        super().__init__()
        self.layers = nn.ModuleList([TransformerEncoderBlock(emb_size) for _ in range(depth)])

    def forward(self, x, mask=None):
        for layer in self.layers:
            x = layer(x, mask=mask)
        return x


class CTCHead(nn.Module):
    def __init__(self, emb_size, num_classes, ffn_hidden=256, dropout=0.3):
        super().__init__()
        self.ffn = nn.Sequential(
            nn.LayerNorm(emb_size),
            nn.Linear(emb_size, ffn_hidden), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(ffn_hidden, num_classes))

    def forward(self, x):
        return self.ffn(x).permute(1, 0, 2)  # (T', B, C)


class DBConformer(nn.Module):
    def __init__(self, args, emb_size=128, tem_depth=6, chn_depth=6, chn=512, n_classes=41):
        super().__init__()
        self.embedding = PatchEmbeddingTemporal(
            data_name=args.data_name, in_planes=args.chn, out_planes=emb_size,
            kernel_size=args.temporal_kernel, radix=1,
            pool_kernel=args.pool_kernel, pool_stride=args.pool_stride,
            time_points=args.time_sample_num, num_classes=args.class_num)
        self.channel_embedding = PatchEmbeddingSpatial(spa_dim=args.spa_dim, emb_size=emb_size)
        self.C = args.chn
        self.posemb_flag = args.posemb_flag
        if args.posemb_flag:
            P_init = max(1, args.time_sample_num // args.pool_stride)
            self.pos_embedding_temporal = nn.Parameter(torch.randn(1, P_init, emb_size))
            self.pos_embedding_spatial  = nn.Parameter(torch.randn(1, self.C, emb_size))
        self.temporal_transformer = TransformerEncoder(tem_depth, emb_size)
        self.spatial_transformer  = TransformerEncoder(chn_depth, emb_size)

    def forward(self, x, mask=None):
        x = x.squeeze(1)                             # (B, C, T)
        x_embed         = self.embedding(x)          # (B, P, D)
        x_embed_spatial = self.channel_embedding(x)  # (B, C, D)
        if self.posemb_flag:
            pos_tem = F.interpolate(
                self.pos_embedding_temporal.permute(0, 2, 1),
                size=x_embed.shape[1], mode='linear', align_corners=False
            ).permute(0, 2, 1)
            x_embed = x_embed + pos_tem
            pos_spa = F.interpolate(
                self.pos_embedding_spatial.permute(0, 2, 1),
                size=x_embed_spatial.shape[1], mode='linear', align_corners=False
            ).permute(0, 2, 1)
            x_embed_spatial = x_embed_spatial + pos_spa
        x_temporal = self.temporal_transformer(x_embed, mask=mask)
        self.spatial_transformer(x_embed_spatial)
        return x_temporal, None


class DBConformerCTC(nn.Module):
    def __init__(self, backbone, emb_size, num_classes, ffn_hidden=256, dropout=0.3):
        super().__init__()
        self.backbone = backbone
        self.ctc_head = CTCHead(emb_size, num_classes, ffn_hidden, dropout)

    def forward(self, x, mask=None):
        features, _ = self.backbone(x, mask=mask)
        return self.ctc_head(features)


def backbone_net_dbconformer(args):
    return DBConformer(
        args, emb_size=args.emb_size,
        tem_depth=args.transformer_depth_tem,
        chn_depth=args.transformer_depth_chn,
        chn=args.chn, n_classes=args.class_num)


print('Model defined.')

In [ ]:
# ================================================================
# Decoding & evaluation helpers
# ================================================================

def greedy_ctc_decode(log_probs):
    tokens = log_probs.argmax(dim=-1).cpu().numpy()  # (T, B)
    decoded = []
    for b in range(tokens.shape[1]):
        seq, prev = [], -1
        for t in range(tokens.shape[0]):
            tok = int(tokens[t, b])
            if tok != 0 and tok != prev:
                seq.append(tok)
            prev = tok
        decoded.append(seq)
    return decoded


def phoneme_error_rate(refs, hyps):
    def lev(a, b):
        n, m = len(a), len(b)
        dp = np.arange(m + 1)
        for i in range(1, n + 1):
            prev = dp.copy(); dp[0] = i
            for j in range(1, m + 1):
                dp[j] = min(prev[j]+1, dp[j-1]+1, prev[j-1]+(0 if a[i-1]==b[j-1] else 1))
        return dp[m]
    total_err = total_len = 0
    for ref, hyp in zip(refs, hyps):
        total_err += lev(ref, hyp)
        total_len += max(len(ref), 1)
    return total_err / total_len * 100.0


def _run_validation(model, val_loader, device, args):
    model.eval()
    all_refs, all_hyps = [], []
    with torch.no_grad():
        for batch_feat, batch_ids, batch_lens, input_time_steps in val_loader:
            batch_feat = batch_feat.to(device)
            input_lengths = compute_token_lengths(
                input_time_steps, args.pool_kernel, args.pool_stride).to(device)
            with torch.autocast(device_type=device.type, dtype=torch.float16):
                log_probs = model(batch_feat).log_softmax(dim=-1)
            hyps = greedy_ctc_decode(log_probs)
            for i in range(len(hyps)):
                slen = int(batch_lens[i].item())
                all_refs.append(batch_ids[i, :slen].tolist())
                all_hyps.append(hyps[i])
    return phoneme_error_rate(all_refs, all_hyps)


def evaluate_with_stats(model, val_loader, device, args):
    """Full evaluation with PER distribution plot."""
    model.eval()
    all_refs, all_hyps, all_pers = [], [], []
    with torch.no_grad():
        for batch_feat, batch_ids, batch_lens, input_time_steps in tqdm(val_loader, desc='Eval'):
            batch_feat = batch_feat.to(device)
            input_lengths = compute_token_lengths(
                input_time_steps, args.pool_kernel, args.pool_stride).to(device)
            with torch.autocast(device_type=device.type, dtype=torch.float16):
                log_probs = model(batch_feat).log_softmax(dim=-1)
            hyps = greedy_ctc_decode(log_probs)
            for i in range(len(hyps)):
                slen = int(batch_lens[i].item())
                ref  = batch_ids[i, :slen].tolist()
                hyp  = hyps[i]
                all_refs.append(ref); all_hyps.append(hyp)
                all_pers.append(phoneme_error_rate([ref], [hyp]))
    pers = np.array(all_pers)
    print(f'Mean PER: {pers.mean():.2f}%  |  Median PER: {np.median(pers):.2f}%')
    plt.figure(figsize=(9, 4))
    plt.hist(pers, bins=20, weights=np.ones(len(pers))/len(pers)*100,
             color='lightblue', edgecolor='white')
    plt.axvline(pers.mean(),     color='red',   linestyle='--', label=f'Mean {pers.mean():.2f}%')
    plt.axvline(np.median(pers), color='green', linestyle=':',  label=f'Median {np.median(pers):.2f}%')
    plt.xlabel('PER (%)'); plt.ylabel('% of trials')
    plt.title('Phoneme Error Rate Distribution'); plt.legend(); plt.tight_layout(); plt.show()
    return pers


print('Helpers defined.')

## training loop

In [ ]:
# ================================================================
# Reusable training loop
# ================================================================

def train_one_epoch(model, optimizer, scheduler, scaler, train_loader, device, args):
    model.train()
    ctc_loss_fn = nn.CTCLoss(blank=0, reduction='mean', zero_infinity=True)
    total_loss, num_batches = 0.0, 0

    for batch_feat, batch_ids, batch_lens, input_time_steps in tqdm(train_loader, leave=False):
        try:
            batch_feat = batch_feat.to(device)
            batch_ids  = batch_ids.to(device)
            batch_lens = batch_lens.to(device)
            input_lengths = compute_token_lengths(
                input_time_steps, args.pool_kernel, args.pool_stride).to(device)

            # Attention mask (B, 1, 1, P)
            B, T_p = batch_feat.size(0), batch_feat.size(-1)
            P = compute_token_lengths(torch.tensor([T_p]), args.pool_kernel, args.pool_stride).item()
            seq_range = torch.arange(P, device=device).expand(B, P)
            attn_mask = (seq_range < input_lengths.unsqueeze(1)).unsqueeze(1).unsqueeze(2)

            optimizer.zero_grad()
            with torch.autocast(device_type=device.type, dtype=torch.float16):
                log_probs = model(batch_feat, mask=attn_mask).log_softmax(dim=-1)

            targets_cat = torch.cat(
                [batch_ids[i, :int(batch_lens[i].item())] for i in range(B)])
            loss = ctc_loss_fn(log_probs, targets_cat, input_lengths, batch_lens)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            if scheduler is not None and not isinstance(
                    scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
                scheduler.step()

            total_loss  += loss.item()
            num_batches += 1

        except torch.cuda.OutOfMemoryError:
            print('\n[OOM] Skipping batch...')
            torch.cuda.empty_cache()
            optimizer.zero_grad()

    return total_loss / num_batches if num_batches > 0 else float('inf')


print('Training loop defined.')

In [ ]:
# ================================================================
# Load data
# ================================================================
gc.collect(); torch.cuda.empty_cache()

train_index = load_all_files(DRIVE_DATA_DIR, LOCAL_DATA_DIR, 'train', MAX_FILES)
val_index   = load_all_files(DRIVE_DATA_DIR, LOCAL_DATA_DIR, 'val',   MAX_FILES)

train_dataset = SpeechDataset(train_index, max_time_steps=args.max_time_steps)
val_dataset   = SpeechDataset(val_index,   max_time_steps=args.max_time_steps)

train_sampler = BucketBatchSampler(train_dataset, batch_size=args.batch_size)
val_sampler   = BucketBatchSampler(val_dataset,   batch_size=args.batch_size)

train_loader = DataLoader(train_dataset, batch_sampler=train_sampler,
                          collate_fn=collate_fn, num_workers=2,
                          pin_memory=True, prefetch_factor=2)
val_loader   = DataLoader(val_dataset,   batch_sampler=val_sampler,
                          collate_fn=collate_fn, num_workers=2,
                          pin_memory=True, prefetch_factor=2)

print(f'Train: {len(train_dataset)} trials, {len(train_loader)} batches')
print(f'Val  : {len(val_dataset)} trials, {len(val_loader)} batches')

## Phase 1 — Train from scratch (OneCycleLR, 100 epochs)

Builds the model from random weights. Best checkpoint saved to `SAVE_DIR_P1` whenever val PER improves.

In [ ]:
gc.collect(); torch.cuda.empty_cache()

backbone_p1 = backbone_net_dbconformer(args).to(device)
model_p1    = DBConformerCTC(
    backbone_p1, emb_size=args.emb_size, num_classes=NUM_PHONEMES,
    ffn_hidden=args.ffn_hidden, dropout=args.dropoutRate).to(device)
print(f'Trainable params: {sum(p.numel() for p in model_p1.parameters() if p.requires_grad):,}')

optimizer_p1 = torch.optim.AdamW(model_p1.parameters(), lr=args.lr, weight_decay=1e-4)
scheduler_p1 = torch.optim.lr_scheduler.OneCycleLR(
    optimizer_p1, max_lr=args.lr,
    steps_per_epoch=len(train_loader), epochs=args.max_epoch)
scaler_p1 = torch.amp.GradScaler('cuda' if device.type == 'cuda' else 'cpu')

os.makedirs(f'{SAVE_DIR_P1}/{args.data_name}/', exist_ok=True)
best_per_p1 = 100.0
best_ckpt_p1 = None

print(f'\n--- Phase 1 ({args.max_epoch} epochs) ---')
for epoch in range(1, args.max_epoch + 1):
    train_loss = train_one_epoch(model_p1, optimizer_p1, scheduler_p1, scaler_p1,
                                  train_loader, device, args)
    if epoch % args.eval_interval == 0 or epoch == args.max_epoch:
        per = _run_validation(model_p1, val_loader, device, args)
        lr  = optimizer_p1.param_groups[0]['lr']
        print(f'Epoch {epoch:03d}/{args.max_epoch} | LR: {lr:.2e} | '
              f'Loss: {train_loss:.4f} | Val PER: {per:.2f}%')
        if per < best_per_p1:
            best_per_p1  = per
            best_ckpt_p1 = f'{SAVE_DIR_P1}/{args.data_name}/DBConformerCTC_{args.session}_p1_best.ckpt'
            torch.save(model_p1.state_dict(), best_ckpt_p1)
            print(f'  → Saved  (PER={best_per_p1:.2f}%)')

print(f'\nPhase 1 complete. Best PER: {best_per_p1:.2f}%')
print(f'Best checkpoint: {best_ckpt_p1}')

## Phase 2 — Continue from Phase 1 best (lower OneCycleLR peak, 50 epochs)

Resumes from the Phase 1 best checkpoint with a reduced peak LR (2.5e-4) to refine without overshooting. If you're restarting the runtime, set `best_ckpt_p1` manually before running this cell.

In [ ]:
# If restarting runtime, set this manually:
# best_ckpt_p1 = '/content/drive/MyDrive/DBConformer_runs_512/NeuralPhoneme/DBConformerCTC_t15.2023.08.18_p1_best.ckpt'

gc.collect(); torch.cuda.empty_cache()

P2_EPOCHS   = 50
P2_MAX_LR   = 2.5e-4

backbone_p2 = backbone_net_dbconformer(args).to(device)
model_p2    = DBConformerCTC(
    backbone_p2, emb_size=args.emb_size, num_classes=NUM_PHONEMES,
    ffn_hidden=args.ffn_hidden, dropout=args.dropoutRate).to(device)
model_p2.load_state_dict(torch.load(best_ckpt_p1, map_location=device))
print(f'Loaded: {best_ckpt_p1}')

optimizer_p2 = torch.optim.AdamW(model_p2.parameters(), lr=5e-4, weight_decay=1e-4)
scheduler_p2 = torch.optim.lr_scheduler.OneCycleLR(
    optimizer_p2, max_lr=P2_MAX_LR,
    steps_per_epoch=len(train_loader), epochs=P2_EPOCHS)
scaler_p2 = torch.amp.GradScaler('cuda' if device.type == 'cuda' else 'cpu')

os.makedirs(f'{SAVE_DIR_P2}/{args.data_name}/', exist_ok=True)
best_per_p2  = best_per_p1
best_ckpt_p2 = best_ckpt_p1   # fallback: keep P1 best if P2 doesn't improve

print(f'\n--- Phase 2 ({P2_EPOCHS} epochs, starting from PER={best_per_p1:.2f}%) ---')
for epoch in range(1, P2_EPOCHS + 1):
    train_loss = train_one_epoch(model_p2, optimizer_p2, scheduler_p2, scaler_p2,
                                  train_loader, device, args)
    if epoch % args.eval_interval == 0 or epoch == P2_EPOCHS:
        per = _run_validation(model_p2, val_loader, device, args)
        lr  = optimizer_p2.param_groups[0]['lr']
        print(f'Epoch {epoch:03d}/{P2_EPOCHS} | LR: {lr:.2e} | '
              f'Loss: {train_loss:.4f} | Val PER: {per:.2f}%')
        if per < best_per_p2:
            best_per_p2  = per
            best_ckpt_p2 = f'{SAVE_DIR_P2}/{args.data_name}/DBConformerCTC_{args.session}_p2_best.ckpt'
            torch.save(model_p2.state_dict(), best_ckpt_p2)
            print(f'  → Saved  (PER={best_per_p2:.2f}%)')

print(f'\nPhase 1 best: {best_per_p1:.2f}%  →  Phase 2 best: {best_per_p2:.2f}%')
print(f'Best checkpoint: {best_ckpt_p2}')

## Phase 3 — Stable fine-tuning (ReduceLROnPlateau, 100 epochs)

No warm-up oscillation. LR halves automatically whenever val PER stops improving for 8 consecutive eval intervals (≈ 40 epochs patience). If you're restarting the runtime, set `best_ckpt_p2` manually.

In [ ]:
# If restarting runtime, set this manually:
# best_ckpt_p2 = '/content/drive/MyDrive/DBConformer_runs_512_p2/NeuralPhoneme/DBConformerCTC_t15.2023.08.18_p2_best.ckpt'

gc.collect(); torch.cuda.empty_cache()

P3_EPOCHS    = 100
P3_DROPOUT   = 0.2   # slightly lower dropout for stable convergence

args.dropoutRate = P3_DROPOUT
backbone_p3 = backbone_net_dbconformer(args).to(device)
model_p3    = DBConformerCTC(
    backbone_p3, emb_size=args.emb_size, num_classes=NUM_PHONEMES,
    ffn_hidden=args.ffn_hidden, dropout=P3_DROPOUT).to(device)
model_p3.load_state_dict(torch.load(best_ckpt_p2, map_location=device))
print(f'Loaded: {best_ckpt_p2}')

optimizer_p3 = torch.optim.AdamW(model_p3.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler_p3 = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_p3, mode='min', factor=0.5, patience=8, verbose=True)
scaler_p3 = torch.amp.GradScaler('cuda' if device.type == 'cuda' else 'cpu')

os.makedirs(f'{SAVE_DIR_P3}/{args.data_name}/', exist_ok=True)
best_per_p3  = best_per_p2
best_ckpt_p3 = best_ckpt_p2

print(f'\n--- Phase 3 ({P3_EPOCHS} epochs, starting from PER={best_per_p2:.2f}%) ---')
for epoch in range(1, P3_EPOCHS + 1):
    train_loss = train_one_epoch(model_p3, optimizer_p3, None, scaler_p3,
                                  train_loader, device, args)
    if epoch % args.eval_interval == 0 or epoch == P3_EPOCHS:
        per = _run_validation(model_p3, val_loader, device, args)
        lr  = optimizer_p3.param_groups[0]['lr']
        print(f'Epoch {epoch:03d}/{P3_EPOCHS} | LR: {lr:.2e} | '
              f'Loss: {train_loss:.4f} | Val PER: {per:.2f}%')
        scheduler_p3.step(per)
        if per < best_per_p3:
            best_per_p3  = per
            best_ckpt_p3 = f'{SAVE_DIR_P3}/{args.data_name}/DBConformerCTC_{args.session}_p3_best.ckpt'
            torch.save(model_p3.state_dict(), best_ckpt_p3)
            print(f'  → Saved  (PER={best_per_p3:.2f}%)')

print(f'\nP1: {best_per_p1:.2f}%  P2: {best_per_p2:.2f}%  P3: {best_per_p3:.2f}%')
print(f'Final best checkpoint: {best_ckpt_p3}')

## Final evaluation

Loads the best checkpoint across all phases and plots the PER distribution.

In [ ]:
# Automatically uses the best checkpoint from whichever phase ran last.
# To evaluate a specific checkpoint, set best_ckpt_p3 manually:
# best_ckpt_p3 = '/content/drive/MyDrive/.../DBConformerCTC_..._p3_best.ckpt'

args.dropoutRate = 0.2
backbone_eval = backbone_net_dbconformer(args).to(device)
model_eval    = DBConformerCTC(
    backbone_eval, emb_size=args.emb_size, num_classes=NUM_PHONEMES,
    ffn_hidden=args.ffn_hidden, dropout=args.dropoutRate).to(device)

final_ckpt = best_ckpt_p3 if 'best_ckpt_p3' in dir() else best_ckpt_p2 if 'best_ckpt_p2' in dir() else best_ckpt_p1
model_eval.load_state_dict(torch.load(final_ckpt, map_location=device))
print(f'Evaluating: {final_ckpt}')

pers = evaluate_with_stats(model_eval, val_loader, device, args)

# Uncertainty

In [ ]:
# ================================================================
# DBConformer v2 AM Uncertainty Analysis
# prerequisite: model_eval (DBConformerCTC) has been loaded, val_loader has been constructed
# ================================================================


# ── 1. AM sequence score ──────────────────────────────────────────
def compute_am_sequence_score(log_probs, input_lengths):
    """
    Per-sample AM uncertainty = negative mean of max log-prob
    over valid CTC frames. Higher value → less confident.

    log_probs     : (T, B, C)  log-softmax output
    input_lengths : (B,)       valid frame counts after pooling
    Returns       : Tensor (B,)
    """
    max_log_probs = log_probs.max(dim=-1)[0]  # (T, B)
    scores = []
    for i in range(log_probs.shape[1]):
        valid_len = int(input_lengths[i].item())
        score = max_log_probs[:valid_len, i].mean().item()
        scores.append(-score)
    return torch.tensor(scores)


# ── 2. Full evaluation loop ───────────────────────────────────────
def evaluate_am_uncertainty_db(model, val_loader, device, args):
    """
    Runs the full val set and collects per-sample PER and AM uncertainty.
    """
    model.eval()
    k_w, s_w = args.pool_kernel, args.pool_stride

    all_pers    = []
    all_am_uncs = []
    samples     = []

    with torch.no_grad():
        for batch_feat, batch_ids, batch_lens, input_time_steps in tqdm(val_loader, desc='AM Uncertainty'):
            batch_feat    = batch_feat.to(device)
            input_lengths = compute_token_lengths(
                input_time_steps, k_w, s_w).to(device)

            # Build attention mask (B, 1, 1, P)
            B, T_p = batch_feat.size(0), batch_feat.size(-1)
            P = compute_token_lengths(
                torch.tensor([T_p]), k_w, s_w).item()
            seq_range = torch.arange(P, device=device).expand(B, P)
            attn_mask = (seq_range < input_lengths.unsqueeze(1)).unsqueeze(1).unsqueeze(2)

            with torch.autocast(device_type=device.type, dtype=torch.float16):
                log_probs = model(batch_feat, mask=attn_mask).log_softmax(dim=-1)

            log_probs = log_probs.float()

            # AM uncertainty
            am_unc = compute_am_sequence_score(log_probs, input_lengths)

            # Greedy decode + PER
            hyps = greedy_ctc_decode(log_probs)
            for i in range(len(hyps)):
                slen = int(batch_lens[i].item())
                ref  = batch_ids[i, :slen].tolist()
                hyp  = hyps[i]
                per  = levenshtein(ref, hyp) / max(len(ref), 1) * 100

                all_pers.append(per)
                all_am_uncs.append(am_unc[i].item())

                if len(samples) < 15:
                    samples.append({
                        'am_unc': am_unc[i].item(),
                        'per': per,
                        'ref': ref[:6],
                        'hyp': hyp[:6],
                    })

    # ── Print sample table ────────────────────────────────────────
    print(f"\n{'AM_UNC':<10} | {'PER':<8} | {'Reference (first 6)':<35} | Hypothesis (first 6)")
    print('-' * 95)
    for s in samples:
        ref_str = ' '.join(LOGIT_TO_PHONEME[p] for p in s['ref'] if p < NUM_PHONEMES)
        hyp_str = ' '.join(LOGIT_TO_PHONEME[p] for p in s['hyp'] if p < NUM_PHONEMES)
        print(f"{s['am_unc']:.4f}    | {s['per']:6.2f}% | {ref_str:<35} | {hyp_str}")

    # ── Correlation ───────────────────────────────────────────────
    corr = np.corrcoef(all_am_uncs, all_pers)[0, 1]
    print(f'\nAM Uncertainty vs PER correlation: {corr:.3f}')
    print(f'Mean PER          : {np.mean(all_pers):.2f}%')
    print(f'Mean AM uncertainty: {np.mean(all_am_uncs):.4f}')

    # ── Confidence stratification ─────────────────────────────────
    q25, q50, q75 = np.percentile(all_am_uncs, [25, 50, 75])
    print(f'\n--- AM Confidence Stratification (quartiles: {q25:.3f} / {q50:.3f} / {q75:.3f}) ---')
    for lo, hi, label in [(0, q25, 'HIGH    '), (q25, q50, 'MEDIUM  '),
                          (q50, q75, 'LOW     '), (q75, 9999, 'VERY_LOW')]:
        grp = [p for p, u in zip(all_pers, all_am_uncs) if lo <= u < hi]
        if grp:
            print(f'{label}: {len(grp):4d} samples ({len(grp)/len(all_pers):.1%}),  '
                  f'mean PER={np.mean(grp):.2f}%,  median PER={np.median(grp):.2f}%')

    # ── Plots ─────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    fig.suptitle('DBConformer v2 — AM Uncertainty Analysis', fontsize=13)

    # Calibration curve
    sorted_idx = np.argsort(all_am_uncs)
    s_pers = [all_pers[i] for i in sorted_idx]
    s_uncs = [all_am_uncs[i] for i in sorted_idx]
    n_bins = 10
    bin_size = max(len(s_pers) // n_bins, 1)
    bin_pers = [np.mean(s_pers[i*bin_size:(i+1)*bin_size]) for i in range(n_bins)]
    bin_uncs = [np.mean(s_uncs[i*bin_size:(i+1)*bin_size]) for i in range(n_bins)]
    axes[0].plot(bin_uncs, bin_pers, 'o-', color='red')
    axes[0].set_xlabel('AM Uncertainty'); axes[0].set_ylabel('Average PER (%)')
    axes[0].set_title('AM Calibration Curve'); axes[0].grid(True, alpha=0.3)

    # Scatter
    axes[1].scatter(all_am_uncs, all_pers, alpha=0.3, s=15, color='green')
    axes[1].set_xlabel('AM Uncertainty'); axes[1].set_ylabel('PER (%)')
    axes[1].set_title(f'AM Uncertainty vs PER  (r={corr:.3f})')
    axes[1].grid(True, alpha=0.3)

    # PER distribution by confidence level
    high_pers = [p for p, u in zip(all_pers, all_am_uncs) if u < q25]
    low_pers  = [p for p, u in zip(all_pers, all_am_uncs) if u >= q75]
    axes[2].hist(high_pers, bins=20, alpha=0.6, color='green',  label=f'High conf (u<{q25:.2f})')
    axes[2].hist(low_pers,  bins=20, alpha=0.6, color='orange', label=f'Low conf  (u>{q75:.2f})')
    axes[2].set_xlabel('PER (%)'); axes[2].set_ylabel('Count')
    axes[2].set_title('PER by Confidence Level'); axes[2].legend()

    plt.tight_layout()
    plt.savefig('/content/drive/MyDrive/DBConformer_AM_uncertainty.png', dpi=150)
    plt.show()

    return all_pers, all_am_uncs


# ── 3. Run ────────────────────────────────────────────────────────
all_pers, all_am_uncs = evaluate_am_uncertainty_db(
    model_eval, val_loader, device, args
)